# U10-04 · Tamaños, comunicación e informe visual de la flota

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 10, apartados 10.8 a 10.10*

Ejecuta cada celda con **Mayús + Intro**. Para que funcione en el navegador, la flota simulada tiene **50 máquinas** en lugar de 200: las máquinas 0-24 son las del sensor descalibrado (en el libro, 0-99). Las figuras se parecerán a las del libro, pero no serán idénticas.

In [ ]:
%pip install -q duckdb pyarrow seaborn plotly
import duckdb, seaborn, plotly
print("DuckDB", duckdb.__version__, "· Seaborn", seaborn.__version__,
      "· Plotly", plotly.__version__)

## 0. Datos y plantilla

El generador de la unidad 9 (50 máquinas) y la plantilla `estilo_empresa` del apartado 10.3.

In [ ]:
import numpy as np
import pandas as pd

def generar_lecturas(maquinas=50, horas=24 * 365, semilla=0):
    """Lecturas horarias simuladas de una flota durante un año."""
    rng = np.random.default_rng(semilla)
    ritmo = rng.uniform(0.5, 1.5, maquinas)      # desgaste por hora
    desgaste = rng.uniform(0, 300, maquinas)     # desde la revisión
    filas = []
    for h in range(horas):
        desgaste += ritmo
        temp = 60 + 0.01 * desgaste + rng.normal(0, 2, maquinas)
        if h >= 24 * 273:              # desde octubre, media flota
            temp[: maquinas // 2] += 3  # con sensor descalibrado
        vib = 2 + 0.004 * desgaste + rng.normal(0, 0.5, maquinas)
        kw = 40 + 0.5 * (temp - 60) + rng.normal(0, 3, maquinas)
        z = -9 + 0.006 * desgaste + 0.8 * (vib - 2)
        averia = rng.random(maquinas) < 1 / (1 + np.exp(-z))
        filas.append((np.full(maquinas, h), np.arange(maquinas),
                      temp, vib, kw, desgaste.copy(), averia))
        desgaste[averia] = 0                      # se repara
    h, m, t, v, k, d, a = map(np.concatenate, zip(*filas))
    df = pd.DataFrame({
        "fecha": pd.Timestamp("2025-01-01") + pd.to_timedelta(h, "h"),
        "maquina": m, "temperatura": t, "vibracion": v,
        "consumo_kw": k, "horas_desde_revision": d, "averia": a})
    # objetivo: ¿se averiará la máquina en las próximas 24 horas?
    df = df.sort_values(["maquina", "fecha"], ignore_index=True)
    def en_24h(s):                               # ¿avería en las 24 h
        futuro = s[::-1].rolling(24, 1).max()[::-1]   # siguientes?
        return futuro.shift(-1, fill_value=0)
    proxima = df.groupby("maquina")["averia"].transform(en_24h)
    df["averia_24h"] = proxima.astype(bool)
    return df

In [ ]:
lecturas = generar_lecturas()
lecturas = lecturas.astype({"maquina": "int16", "temperatura": "float32",
    "vibracion": "float32", "consumo_kw": "float32",
    "horas_desde_revision": "float32"})
lecturas.to_parquet("lecturas.parquet", index=False)
print(len(lecturas), "lecturas guardadas en lecturas.parquet")

In [ ]:
import matplotlib as mpl
from cycler import cycler

def estilo_empresa():
    """Aspecto común para todos los gráficos de la empresa."""
    mpl.rcParams.update({
        "font.family": "sans-serif", "font.size": 7,
        "figure.figsize": (4.4, 2.6),          # 11 cm de ancho
        "axes.prop_cycle": cycler(color=[
            "#1F5FA8", "#E08A1E", "#7FA7D6", "#14304F", "#9AA5B1"]),
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": "#E3E8EE",
        "lines.linewidth": 1.2, "lines.markersize": 3.5,
        "grid.linewidth": 0.6, "axes.axisbelow": True,
        "axes.titlesize": 7.5, "axes.titleweight": "bold",
        "axes.titlelocation": "left", "legend.frameon": False,
        "savefig.bbox": "tight", "pdf.fonttype": 42})

## 1. Cuánto ocupa la misma figura (ejemplo 4)

In [ ]:
import io
import pandas as pd
import matplotlib.pyplot as plt

estilo_empresa()
d = pd.read_parquet("lecturas.parquet")
mes = d.groupby(d["fecha"].dt.month)["averia"].sum()
nube = d.sample(100_000, random_state=0)

def figura(tipo):
    fig, eje = plt.subplots(figsize=(6, 3.5))
    if tipo == "barras":
        eje.bar(mes.index, mes.values)
    else:
        eje.scatter(nube["horas_desde_revision"],
                    nube["vibracion"], s=1, alpha=0.2)
    plt.close(fig)
    return fig

def kb(fig, **opciones):
    buf = io.BytesIO()
    fig.savefig(buf, **opciones)
    return buf.tell() / 1024

formatos = {"PNG 72 ppp": dict(format="png", dpi=72),
            "PNG 150 ppp": dict(format="png", dpi=150),
            "PNG 300 ppp": dict(format="png", dpi=300),
            "JPEG 150 ppp": dict(format="jpg", dpi=150),
            "WebP 150 ppp": dict(format="webp", dpi=150),
            "SVG": dict(format="svg"), "PDF": dict(format="pdf")}
tabla = pd.DataFrame({t: {f: kb(figura(t), **o)
                          for f, o in formatos.items()}
                      for t in ["barras", "nube"]})
print(tabla.round(0))

# energía de servir la figura 10 000 veces (0,06 kWh/GB, 2015)
gb = tabla * 10_000 / 1024**2
print((gb * 0.06).round(3))

**Prueba.** Añade `"WebP 150 ppp sin pérdida": dict(format="webp", dpi=150, pil_kwargs={"lossless": True})`.

## 2. Antes y después

In [ ]:
import duckdb
import matplotlib.pyplot as plt

estilo_empresa()
t = duckdb.sql("""
    SELECT month(fecha) AS mes,
        avg(temperatura) FILTER (maquina < 25)  AS afectada,
        avg(temperatura) FILTER (maquina >= 25) AS sin_cambio
    FROM 'lecturas.parquet' GROUP BY mes ORDER BY mes
""").df()
print(t.round(2).to_string(index=False))

fig, (antes, despues) = plt.subplots(2, 1, figsize=(4.4, 4.2))
antes.plot(t["mes"], t["afectada"], label="afectada")
antes.plot(t["mes"], t["sin_cambio"], label="sin_cambio")
antes.legend()
antes.set_title("Temperatura media mensual")

despues.plot(t["mes"], t["sin_cambio"], color="#9AA5B1")
despues.plot(t["mes"], t["afectada"], color="C1", lw=2.5)
despues.text(12.3, t["afectada"].iloc[-1], "máquinas 0-24",
             color="C1", va="center")
despues.text(12.3, t["sin_cambio"].iloc[-1], "máquinas 25-49",
             color="#7A8591", va="center")
despues.axvspan(9.5, 12.5, color="C1", alpha=0.08, lw=0)
despues.set_title("Desde octubre, la mitad de la flota marca"
                  " 3 °C más:\nrevisar sus sensores")
despues.set(xlabel="Mes", ylabel="Temperatura media (°C)",
            xticks=range(1, 13), xlim=(0.5, 12.5))
despues.spines[["left"]].set_visible(False)
fig.tight_layout()
fig.savefig("comunicar.pdf")

## 3. El informe de la flota (ejemplo 5)

In [ ]:
import os
import duckdb
import matplotlib.pyplot as plt
import plotly.express as px

estilo_empresa()
q = lambda sql: duckdb.sql(sql).df()
mes = q("""SELECT month(fecha) AS mes,
               count_if(averia)::INTEGER AS averias
           FROM 'lecturas.parquet' GROUP BY mes ORDER BY mes""")
riesgo = q("""SELECT floor(horas_desde_revision / 50) * 50 + 25
                   AS horas, avg(averia_24h::INTEGER) AS p
              FROM 'lecturas.parquet'
              WHERE horas_desde_revision < 600
              GROUP BY horas ORDER BY horas""")
temp = q("""SELECT month(fecha) AS mes,
             avg(temperatura) FILTER (maquina < 25) AS afectada,
             avg(temperatura) FILTER (maquina >= 25) AS resto
            FROM 'lecturas.parquet' GROUP BY mes ORDER BY mes""")

fig, ejes = plt.subplot_mosaic([[0, 1], [2, 2]],
                               figsize=(4.4, 4.0))
ejes[0].bar(mes["mes"], mes["averias"], color="#9AA5B1")
ejes[0].set(title="Unas 370 averías al mes,\nsin estacionalidad",
            xticks=range(1, 13, 2), xlabel="Mes", ylabel="Averías")
ejes[1].plot(riesgo["horas"], 100 * riesgo["p"], "o-")
ejes[1].axvline(300, color="C1", ls="--")
ejes[1].set(title="El riesgo se dispara a partir\nde unas 300 horas",
            xlabel="Horas desde la revisión",
            ylabel="% con avería en 24 h")
ejes[2].plot(temp["mes"], temp["resto"], color="#9AA5B1")
ejes[2].plot(temp["mes"], temp["afectada"], color="C1", lw=2.5)
ejes[2].set(title="Sensores de las máquinas 0-24\ndescalibrados"
            " desde octubre", xticks=range(1, 13), xlabel="Mes",
            ylabel="Temperatura media (°C)")
fig.tight_layout()
for ext, opc in [("pdf", {}), ("png", {"dpi": 150})]:
    fig.savefig(f"informe_flota.{ext}", **opc)

maq = q("""SELECT maquina, count_if(averia)::INTEGER AS averias,
               round(avg(horas_desde_revision)) AS horas_medias
           FROM 'lecturas.parquet' GROUP BY maquina""")
px.scatter(maq, x="horas_medias", y="averias",
           hover_name="maquina",
           title="Averías por máquina (pasa el ratón)").write_html(
    "informe_flota.html", include_plotlyjs="cdn")
for f in ["informe_flota.pdf", "informe_flota.png",
          "informe_flota.html"]:
    print(f, f"{os.path.getsize(f) / 1024:.0f} KB")
print(riesgo.round(3).to_string(index=False))

In [ ]:
import plotly.express as px
from IPython.display import HTML
interactivo = px.scatter(maq, x="horas_medias", y="averias",
                         hover_name="maquina")
HTML(interactivo.to_html(include_plotlyjs="cdn"))

**Ejercicio.** Cambia el título de cada panel por uno que solo describa («Averías por mes»…) y compara cuál entiendes antes.

> Aslan, J., Mayers, K., Koomey, J. G. y France, C. (2018). Electricity intensity of Internet data transmission. *Journal of Industrial Ecology*, 22(4), 785–798. https://doi.org/10.1111/jiec.12630
>
> Knaflic, C. N. (2015). *Storytelling with Data*. Wiley. ISBN 978-1-119-00225-3